<a href="https://colab.research.google.com/github/NadiaTeta/group1/blob/main/notebooks/04_feed_forward.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q transformers

import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

torch.manual_seed(42)   # so random results are the same for everyone
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# load the already-trained GPT-2 (small version) from Hugging Face
tokenizer = GPT2TokenizerFast.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    output_attentions=True,         # lets us look at attention weights
    output_hidden_states=True,      # lets us look at the numbers after each layer
    attn_implementation="eager",    # needed so attention weights can be returned
).to(device)
model.eval()                        # we're exploring, not training

print("Device:", device)
print(f"GPT-2 loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.0f} million parameters")

# shared example sentences: the same meaning in English and Kinyarwanda
english = "The people of Kicukiro say they have not had water for three days."
kinyarwanda = "Abaturage bo mu Kicukiro bavuga ko bamaze iminsi itatu nta mazi bafite."

for name, sentence in [("English", english), ("Kinyarwanda", kinyarwanda)]:
    tokens = tokenizer.tokenize(sentence)
    print(f"\n{name}: {len(tokens)} tokens")
    print(tokens)



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Device: cpu
GPT-2 loaded: 124 million parameters

English: 17 tokens
['The', 'Ġpeople', 'Ġof', 'ĠK', 'ic', 'uk', 'iro', 'Ġsay', 'Ġthey', 'Ġhave', 'Ġnot', 'Ġhad', 'Ġwater', 'Ġfor', 'Ġthree', 'Ġdays', '.']

Kinyarwanda: 29 tokens
['Ab', 'atur', 'age', 'Ġbo', 'Ġmu', 'ĠK', 'ic', 'uk', 'iro', 'Ġb', 'av', 'uga', 'Ġko', 'Ġb', 'am', 'aze', 'Ġim', 'ins', 'i', 'Ġit', 'atu', 'Ġn', 'ta', 'Ġm', 'azi', 'Ġb', 'af', 'ite', '.']


In [9]:
import torch.nn as nn
from contextlib import contextmanager

# --- 1. Define FFN Muting Context Manager ---
@contextmanager
def mute_mlp(layer_idx):
    # Save the original mlp block's forward pass
    original_mlp_forward = model.transformer.h[layer_idx].mlp.forward

    # Define a replacement that returns zeros (effectively switching it off)
    def dummy_forward(*args, **kwargs):
        output = original_mlp_forward(*args, **kwargs)
        return torch.zeros_like(output)

    # Override the forward pass
    model.transformer.h[layer_idx].mlp.forward = dummy_forward
    try:
        yield
    finally:
        # Restore original forward pass after exiting 'with' block
        model.transformer.h[layer_idx].mlp.forward = original_mlp_forward

# --- 2. Calculate Loss Before & After ---
def compute_loss(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    inputs["labels"] = inputs["input_ids"].clone()
    with torch.no_grad():
        outputs = model(**inputs)
    return outputs.loss.item()

test_layer = 6

print("=== Loss Comparison (Before vs After disabling MLP) ===")
for name, sentence in [("English", english), ("Kinyarwanda", kinyarwanda)]:
    loss_normal = compute_loss(sentence)
    with mute_mlp(test_layer):
        loss_muted = compute_loss(sentence)
    print(f"{name} Sentence:")
    print(f"  Original Loss: {loss_normal:.4f}")
    print(f"  Loss with MLP {test_layer} disabled: {loss_muted:.4f}")
    print(f"  Difference (Increase): {loss_muted - loss_normal:.4f}\n")

# --- 3. Text Generation Before & After ---
prompt = "The key aspect of the feed-forward network is"
prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)

def generate_text():
    with torch.no_grad():
        output = model.generate(
            prompt_ids,
            max_length=30,
            num_return_sequences=1,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            temperature=0.8
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print("=== Text Generation Comparison ===")
# Set random seed for consistency
torch.manual_seed(42)
print("Normal Generation:")
print(f'"{generate_text()}"')

torch.manual_seed(42)
print(f"\nGeneration with MLP Layer {test_layer} Disabled:")
with mute_mlp(test_layer):
    print(f'"{generate_text()}"')


=== Loss Comparison (Before vs After disabling MLP) ===
English Sentence:
  Original Loss: 4.4565
  Loss with MLP 6 disabled: 4.5035
  Difference (Increase): 0.0470

Kinyarwanda Sentence:
  Original Loss: 7.3497
  Loss with MLP 6 disabled: 7.2714
  Difference (Increase): -0.0782

=== Text Generation Comparison ===
Normal Generation:
"The key aspect of the feed-forward network is that there is a finite number of neurons that can be connected to the feed-forward network.
"

Generation with MLP Layer 6 Disabled:
"The key aspect of the feed-forward network is that there is a finite loop in which the feed-forward network moves forward once again. This happens"
